# Monitoring and reports (run in SageMaker Studio)

Covers the data monitor, model monitor, infrastructure monitor, dashboard and reports, with a single cleanup check at the end. Run `3_train_evaluate_deploy.ipynb` (incl. approval and deploy) first. The dashboard's report-index link is populated once the Reports section has run; re-run the Dashboard section afterwards if you want it refreshed.

In [ ]:
# SageMaker Python SDK v3 (default on current Studio images) removed the v2 modules this notebook uses.
# Pin v2; on first run this installs it and restarts the kernel - then run the notebook again from the top.
from importlib.metadata import version, PackageNotFoundError
try:
    _sm_ver = version("sagemaker")
except PackageNotFoundError:
    _sm_ver = "0"
if int(_sm_ver.split(".")[0]) != 2:
    %pip uninstall -q -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
    %pip install -q "sagemaker>=2.200,<3"
    print(f"Replaced sagemaker {_sm_ver} with v2 - restarting kernel; re-run the notebook from the top.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("sagemaker", _sm_ver)

## Bootstrap

Path, config, boto3 session/clients, shared S3/CloudWatch helpers and the one-off Model Monitor analyzer helper.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json, time
import boto3, numpy as np, pandas as pd
from botocore.exceptions import ClientError
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
from src.pamap2 import monitoring as M
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
cw = session.client("cloudwatch")
sm = session.client("sagemaker")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")
MON = lambda key: s3_uri(cfg, "monitoring", key)
NS = cfg["monitoring_namespace"]

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_bytes(uri):
    b, k = split_key(uri)
    return s3.get_object(Bucket=b, Key=k)["Body"].read()
def read_bytes_or_none(uri):
    try:
        return read_bytes(uri)
    except ClientError as e:
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return None
        raise
def read_parquet(name):
    return pd.read_parquet(io.BytesIO(read_bytes(SPLITS + name)))
def read_json(uri):
    return json.loads(read_bytes(uri))
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)
def put_text(uri, text):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=text.encode())
    return uri
def exists(uri):
    b, k = split_key(uri)
    try:
        s3.head_object(Bucket=b, Key=k)
        return True
    except ClientError:
        return False
def latest_subprefix(uri_prefix):
    """Newest child 'folder' (timestamped run folders sort lexicographically)."""
    b, k = split_key(uri_prefix)
    r = s3.list_objects_v2(Bucket=b, Prefix=k, Delimiter="/")
    subs = sorted(p["Prefix"] for p in r.get("CommonPrefixes", []))
    return f"s3://{b}/{subs[-1]}" if subs else None
def publish(metrics):
    """metrics: {name: value} -> custom namespace (put_metric_data)."""
    cw.put_metric_data(Namespace=NS, MetricData=[{"MetricName": n, "Value": float(v), "Unit": "None"} for n, v in metrics.items()])
    print("Published", metrics)

def safe_publish(metrics):
    try:
        publish(metrics)
    except Exception as e:   # never mask the original error
        print("WARNING: publish failed:", e)
def report_failure(job_name):
    try:
        print("FailureReason:", sm.describe_processing_job(ProcessingJobName=job_name).get("FailureReason"))
    except Exception as e:
        print("Could not describe job:", e)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
ACTIVITY_IDS = list(LABEL_MAP.values())
print(K, "classes;", len(FEATURES), "features")

In [ ]:
import sagemaker
from sagemaker.processing import Processor, ProcessingInput, ProcessingOutput
sm_session = sagemaker.Session(boto_session=session)
ANALYZER = sagemaker.image_uris.retrieve("model-monitor", cfg["region"])

def run_analyzer(name, env, inputs, output_uri):
    """One-off Model Monitor analyzer job (ml.m5.large x1). inputs: [(s3_uri, container_dest, input_name)]."""
    proc = Processor(role=cfg["role_arn"], image_uri=ANALYZER, instance_count=1, instance_type="ml.m5.large",
                     volume_size_in_gb=20, max_runtime_in_seconds=3600, env=env, sagemaker_session=sm_session)
    proc.run(inputs=[ProcessingInput(source=s, destination=d, input_name=n) for s, d, n in inputs],
             outputs=[ProcessingOutput(source="/opt/ml/processing/output", destination=output_uri, output_name="result")],
             job_name=name, wait=True, logs=False)
    return output_uri
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

## Data monitor (run in SageMaker Studio)

Data-quality baseline (statistics + constraints) from the **train** features with `DefaultModelMonitor.suggest_baseline`, then a one-off Model Monitor analyzer job on the **production** features. Batch only: no endpoint, no monitoring schedule. Two billable `ml.m5.large` x1 processing jobs. Outputs: `monitoring/data-quality/baseline/` and `monitoring/data-quality/runs/<ts>/`.

### Monitor datasets (header CSV, feature_columns.json order, no label)

In [ ]:
train_uri = put_text(MON("data-quality/input/train.csv"), M.to_monitor_csv(read_parquet("train.parquet"), FEATURES))
prod_df = read_parquet("production.parquet")
prod_uri = put_text(MON("data-quality/input/production.csv"), M.to_monitor_csv(prod_df, FEATURES))
print(train_uri, prod_uri, len(prod_df), "production rows")


### Baseline on train (billable: one ml.m5.large processing job)

In [ ]:
DQ_BASELINE = MON("data-quality/baseline/")
dq_monitor = DefaultModelMonitor(role=cfg["role_arn"], instance_count=1, instance_type="ml.m5.large",
                              volume_size_in_gb=20, max_runtime_in_seconds=3600, sagemaker_session=sm_session)
dq_monitor.suggest_baseline(baseline_dataset=train_uri, dataset_format=DatasetFormat.csv(header=True),
                         output_s3_uri=DQ_BASELINE, wait=True, logs=False)
stats = read_json(DQ_BASELINE + "statistics.json")
cons = read_json(DQ_BASELINE + "constraints.json")
print(len(stats["features"]), "features profiled;", len(cons["features"]), "constraints")
cons_by_name = {c["name"]: c for c in cons["features"]}
display(pd.DataFrame([{"feature": f["name"], "type": f["inferred_type"],
                       "completeness_threshold": cons_by_name.get(f["name"], {}).get("completeness")}
                      for f in stats["features"][:10]]))


### One-off monitoring job on production features (billable: one ml.m5.large processing job)

In [ ]:
dq_stamp = time.strftime("%Y%m%d-%H%M%S")
DQ_RUN = MON(f"data-quality/runs/{dq_stamp}/")
env = {"dataset_format": json.dumps(DatasetFormat.csv(header=True)), "dataset_source": "/opt/ml/processing/input/data",
       "output_path": "/opt/ml/processing/output", "publish_cloudwatch_metrics": "Disabled",
       "baseline_constraints": "/opt/ml/processing/baseline/constraints/constraints.json",
       "baseline_statistics": "/opt/ml/processing/baseline/stats/statistics.json"}
dq_job_name = f"{cfg['prefix']}-data-monitor-{dq_stamp}"
try:
    run_analyzer(dq_job_name, env, [
        (prod_uri, "/opt/ml/processing/input/data", "data"),
        (DQ_BASELINE + "constraints.json", "/opt/ml/processing/baseline/constraints", "constraints"),
        (DQ_BASELINE + "statistics.json", "/opt/ml/processing/baseline/stats", "statistics")], DQ_RUN)
except Exception:
    report_failure(dq_job_name)
    safe_publish({"DataJobFailed": 1})
    raise
safe_publish({"DataJobFailed": 0})


### Statistics, constraints and violations

In [ ]:
assert exists(DQ_RUN + "statistics.json"), "no statistics.json in run output - job incomplete; absent violations would not mean zero"
print("Run statistics:", DQ_RUN + "statistics.json")
print("Run constraints:", DQ_RUN + "constraints.json", "| exists:", exists(DQ_RUN + "constraints.json"))
v = M.parse_violations(read_bytes_or_none(DQ_RUN + "constraint_violations.json"))   # missing file = 0 violations
print("Violations:", v["count"], "->", v["status"])
display(pd.DataFrame(v["violations"], columns=["feature", "check_type", "description"]))
write_json(DQ_RUN + "violations_summary.json", v)
publish({"DataViolationCount": v["count"]})


## Model monitor (run in SageMaker Studio)

`ModelQualityMonitor` (multiclass) baselined on **test** predictions vs labels, then a one-off analyzer job on the **production** predictions vs `true_activity_id`, compared with `xgboost_eval.json`. Batch only: no endpoint or schedule.

**Caveat:** test and production windows come from subjects seen in training (chronological split), so these quality numbers are optimistic.

### Baseline dataset: test predictions (from the evaluation batch-transform output) vs test labels

In [ ]:
test = read_parquet("test.parquet").reset_index(drop=True)
probs = E.parse_probs(read_bytes(s3_uri(cfg, "models", "transform-output/test/features.csv.out")), K)
assert len(probs) == len(test), (len(probs), len(test))
cls = E.probs_to_labels(probs, K)
test_preds = pd.DataFrame({"predicted_activity_id": [LABEL_MAP[int(c)] for c in cls], "confidence": probs.max(axis=1),
                           "true_activity_id": test["activity_id"]})
base_df, dropped = M.to_model_quality_frame(test_preds)
print("Baseline rows:", len(base_df), "| dropped NaN labels:", dropped)
base_uri = put_text(MON("model-quality/input/baseline.csv"), base_df.to_csv(index=False))
base_metrics = E.classification_metrics(base_df["label"], base_df["prediction"], ACTIVITY_IDS)
eval_report = read_json(REPORTS + "xgboost_eval.json")
print(f"Local test macro-F1={base_metrics['macro_f1']:.4f} | xgboost_eval.json macro-F1={eval_report['macro_f1']:.4f}")
if abs(base_metrics["macro_f1"] - eval_report["macro_f1"]) > 1e-3:
    print("WARNING: baseline macro-F1 differs from xgboost_eval.json by > 1e-3 - transform output may be stale")


### Baseline job (billable: one ml.m5.large processing job)

In [ ]:
MQ_BASELINE = MON("model-quality/baseline/")
mq = ModelQualityMonitor(role=cfg["role_arn"], instance_count=1, instance_type="ml.m5.large", volume_size_in_gb=20,
                         max_runtime_in_seconds=3600, sagemaker_session=sm_session)
mq.suggest_baseline(baseline_dataset=base_uri, dataset_format=DatasetFormat.csv(header=True), output_s3_uri=MQ_BASELINE,
                    problem_type="MulticlassClassification", inference_attribute="prediction",
                    probability_attribute="probability", ground_truth_attribute="label", wait=True, logs=False)
for f in ("statistics.json", "constraints.json"):
    print(MQ_BASELINE + f, "exists:", exists(MQ_BASELINE + f))


### Production predictions vs true_activity_id

In [ ]:
pb, pk = split_key(s3_uri(cfg, "models", "predictions/production/"))
objs = [o for p in s3.get_paginator("list_objects_v2").paginate(Bucket=pb, Prefix=pk)
        for o in p.get("Contents", []) if o["Key"].endswith(".parquet")]
if not objs:
    raise RuntimeError("no production prediction parquet found - run the Deploy section first")
newest = max(objs, key=lambda o: o["LastModified"])["Key"]
stem = newest[:-len(".parquet")]
parts = [o for o in objs if o["Key"] == newest or o["Key"].startswith(stem + "/")]   # all parts of the latest Deploy output
print("Using", [o["Key"] for o in parts])
preds = pd.concat([pd.read_parquet(io.BytesIO(s3.get_object(Bucket=pb, Key=o["Key"])["Body"].read())) for o in parts],
                  ignore_index=True)
n_prod = len(read_parquet("production.parquet"))   # split file is already restricted to train classes (= Deploy input)
assert len(preds) == n_prod, f"predictions rows {len(preds)} != production split rows {n_prod}"
prod_df, dropped = M.to_model_quality_frame(preds)
if prod_df.empty:
    raise RuntimeError("no labelled production rows (true_activity_id all NaN)")
print("Production rows:", len(prod_df), "| dropped rows with NaN label/prediction/probability:", dropped)
prod_uri = put_text(MON("model-quality/input/production.csv"), prod_df.to_csv(index=False))
prod_metrics = E.classification_metrics(prod_df["label"], prod_df["prediction"], ACTIVITY_IDS)


### One-off monitoring job (billable: one ml.m5.large processing job)

In [ ]:
mq_stamp = time.strftime("%Y%m%d-%H%M%S")
MQ_RUN = MON(f"model-quality/runs/{mq_stamp}/")
env = {"analysis_type": "MODEL_QUALITY", "problem_type": "MulticlassClassification",
       "dataset_format": json.dumps(DatasetFormat.csv(header=True)), "dataset_source": "/opt/ml/processing/input/data",
       "inference_attribute": "prediction", "probability_attribute": "probability", "ground_truth_attribute": "label",
       "output_path": "/opt/ml/processing/output", "publish_cloudwatch_metrics": "Disabled",
       "baseline_constraints": "/opt/ml/processing/baseline/constraints/constraints.json",
       "baseline_statistics": "/opt/ml/processing/baseline/stats/statistics.json"}
mq_job_name = f"{cfg['prefix']}-model-monitor-{mq_stamp}"
try:
    run_analyzer(mq_job_name, env, [
        (prod_uri, "/opt/ml/processing/input/data", "data"),
        (MQ_BASELINE + "constraints.json", "/opt/ml/processing/baseline/constraints", "constraints"),
        (MQ_BASELINE + "statistics.json", "/opt/ml/processing/baseline/stats", "statistics")], MQ_RUN)
except Exception:
    report_failure(mq_job_name)
    safe_publish({"ModelJobFailed": 1})
    raise
safe_publish({"ModelJobFailed": 0})


### Violations and quality drift vs xgboost_eval.json

In [ ]:
assert exists(MQ_RUN + "statistics.json"), "no statistics.json in run output - job incomplete; absent violations would not mean zero"
v = M.parse_violations(read_bytes_or_none(MQ_RUN + "constraint_violations.json"))
print("Violations:", v["count"], "->", v["status"])
display(pd.DataFrame(v["violations"], columns=["feature", "check_type", "description"]))
drift = M.quality_drift(eval_report["macro_f1"], prod_metrics["macro_f1"], cfg["monitoring_drift_threshold"])
print(drift)
display(pd.DataFrame([{"metric": m, "baseline_eval": eval_report[m], "production": prod_metrics[m]} for m in ("macro_f1", "accuracy")]))
print("NOTE: seen-subject production windows make these numbers optimistic (chronological split).")
write_json(MQ_RUN + "violations_summary.json", {**v, "drift": drift})
publish({"ModelMacroF1": prod_metrics["macro_f1"], "ModelAccuracy": prod_metrics["accuracy"],
         "ModelViolationCount": v["count"], "ModelDegraded": int(drift["degraded"])})


## Infrastructure monitor (run in SageMaker Studio)

CloudWatch alarms on SageMaker job utilisation (`/aws/sagemaker/TrainingJobs|TransformJobs|ProcessingJobs`: CPU, memory, disk) plus alarms on the custom `pamap2-har/Monitoring` metrics published by the data and model monitors (violation counts, degradation, job-failed 0/1). No SNS topic or EventBridge rule (user decision): alarm state is shown here and on the dashboard. Alarm creation is idempotent (`put_metric_alarm` overwrites). Job alarms are Metrics Insights query alarms over the whole job namespace (one query per job namespace and metric), so future jobs are watched too. Custom-metric alarms use `TreatMissingData=ignore` so the last evaluated state persists between manual runs.

In [ ]:
def put_alarm(spec):
    """put_metric_alarm; if AccessDenied on actions, retry without AlarmActions (none are configured by default)."""
    try:
        cw.put_metric_alarm(**spec)
    except ClientError as e:
        if e.response["Error"]["Code"] != "AccessDenied" or "AlarmActions" not in spec:
            raise
        print("WARNING: AccessDenied for alarm actions; creating", spec["AlarmName"], "without actions")
        cw.put_metric_alarm(**{k: v for k, v in spec.items() if k != "AlarmActions"})


### Job-utilisation alarms

In [ ]:
prefix = cfg["monitoring_alarm_prefix"]
created = []
THRESHOLDS = {"CPUUtilization": 95, "MemoryUtilization": 90, "DiskUtilization": 85}
for job_type in M.JOB_NAMESPACES:
    for metric, thr in THRESHOLDS.items():
        spec = M.alarm_spec(job_type, metric, thr, prefix=prefix)   # Metrics Insights query: covers every current and future job
        put_alarm(spec); created.append(spec["AlarmName"])
print("Created/updated:", created)


### Monitoring-metric alarms (custom namespace)

In [ ]:
for metric, thr in {"DataJobFailed": 0, "ModelJobFailed": 0, "DataViolationCount": 0,
                    "ModelViolationCount": 0, "ModelDegraded": 0}.items():
    spec = M.alarm_spec("custom", metric, thr, prefix=prefix, namespace=NS, period=3600)
    put_alarm(spec); print("Created/updated", spec["AlarmName"])


### Alarm states

In [ ]:
alarms = cw.describe_alarms(AlarmNamePrefix=prefix + "-")["MetricAlarms"]
def _alarm_metric(a):
    # Metric-math alarms have no top-level MetricName; show their expression / metric names instead.
    if "MetricName" in a:
        return a["MetricName"]
    return ", ".join(m.get("Expression") or m.get("MetricStat", {}).get("Metric", {}).get("MetricName", m["Id"])
                     for m in a.get("Metrics", []))
display(pd.DataFrame([{"alarm": a["AlarmName"], "state": a["StateValue"], "metric": _alarm_metric(a)}
                      for a in alarms]))


### Cleanup

This notebook creates only alarms (kept on purpose, free tier). The final Cleanup section at the end of this notebook verifies nothing else is running.

## Dashboard (run in SageMaker Studio)

Creates/overwrites the `pamap2-har-ml` CloudWatch dashboard: job utilisation widgets, custom monitoring metrics, alarm status and a text widget linking the reports and the Debugger report. Run after the Infrastructure monitor and ideally the Reports section.

In [ ]:
job_metrics = [{"title": f"{jt} {m}", "namespace": ns, "metric": m}
               for jt, ns in M.JOB_NAMESPACES.items() for m in ("CPUUtilization", "MemoryUtilization")]
alarm_arns = [a["AlarmArn"] for a in cw.describe_alarms(AlarmNamePrefix=cfg["monitoring_alarm_prefix"] + "-")["MetricAlarms"]]
links = {"report index (populated after Reports)": MON("reports/index.json")}
try:
    links["Debugger report"] = read_json(REPORTS + "xgboost_training.json")["debugger_artifacts"] + M.DEBUGGER_REPORT_PATH
except Exception as e:
    print("No Debugger path:", e)
body = M.dashboard_body(cfg["region"], job_metrics,
                        ["DataViolationCount", "ModelViolationCount", "ModelMacroF1", "ModelAccuracy", "ModelDegraded",
                         "DataJobFailed", "ModelJobFailed"], alarm_arns, NS, links)
resp = cw.put_dashboard(DashboardName=cfg["monitoring_dashboard"], DashboardBody=body)
print("Validation messages:", resp.get("DashboardValidationMessages"))
print(f"https://{cfg['region']}.console.aws.amazon.com/cloudwatch/home?region={cfg['region']}#dashboards:name={cfg['monitoring_dashboard']}")


### Cleanup

The dashboard is intentionally kept. The final Cleanup section at the end of this notebook verifies nothing else is running.

## Reports (run in SageMaker Studio)

**SHAP explainability** for the evaluated XGBoost model, then a consolidated index `monitoring/reports/index.json` of the data-quality and model-quality constraints/violations, the SHAP outputs and the Debugger report.

**Why not Clarify:** SageMaker Clarify processing is in maintenance mode and rejects `CreateProcessingJob` for new accounts. SHAP is computed here instead with XGBoost's built-in exact TreeSHAP (`pred_contribs=True`) on the registered model artifact - no job, endpoint or model is created, so this notebook is not billable. SHAP on test windows from seen subjects is still optimistic.

In [ ]:
# Local exact TreeSHAP needs the xgboost library matching the training container (cfg["xgboost_version"], e.g. 1.7-1).
XGB_PIN = "xgboost==" + cfg["xgboost_version"].split("-")[0] + ".*"
try:
    import xgboost as xgb
except ImportError:
    %pip install -q "{XGB_PIN}"
    import xgboost as xgb
print("xgboost", xgb.__version__)
train_info = read_json(REPORTS + "xgboost_training.json")
PKG_ARN = read_json(REPORTS + "xgboost_eval.json")["model_package_arn"]   # the evaluated package
SHAP_OUT = MON("reports/shap/")

### SHAP explainability (local exact TreeSHAP - not billable)

In [ ]:
import pickle, tarfile
test = read_parquet("test.parquet").reset_index(drop=True)
sample = test.sample(n=min(200, len(test)), random_state=0)

# Model artifact of the evaluated package -> Booster
model_data = sm.describe_model_package(ModelPackageName=PKG_ARN)["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]
with tarfile.open(fileobj=io.BytesIO(read_bytes(model_data)), mode="r:gz") as tf:
    raw = tf.extractfile(next(m for m in tf.getmembers() if m.isfile())).read()
booster = xgb.Booster()
try:
    booster.load_model(bytearray(raw))   # built-in algorithm's native format
except xgb.core.XGBoostError:
    booster = pickle.loads(raw)          # older containers pickle the Booster
print("Model:", model_data)

# Exact TreeSHAP; multiclass -> (n, K, F+1), last column is the bias term. NaNs use the model's missing-value paths.
contribs = booster.predict(xgb.DMatrix(sample[FEATURES].to_numpy(float)), pred_contribs=True)
assert contribs.shape == (len(sample), K, len(FEATURES) + 1), contribs.shape
mean_abs = np.abs(contribs[:, :, :-1]).mean(axis=0)               # (K, F)
class_names = [ACTIVITY_NAMES[LABEL_MAP[i]] for i in range(K)]
global_imp = pd.Series(mean_abs.mean(axis=0), index=FEATURES).sort_values(ascending=False)
per_class = pd.DataFrame(mean_abs.T, index=FEATURES, columns=class_names)

shap_dir = SHAP_OUT + time.strftime("%Y%m%d-%H%M%S") + "/"
write_json(shap_dir + "analysis.json", {
    "method": "xgboost exact TreeSHAP (pred_contribs)", "model_package_arn": PKG_ARN, "model_data": model_data,
    "n_samples": len(sample), "agg_method": "mean_abs",
    "global_shap_values": global_imp.to_dict(),
    "per_class_shap_values": {c: per_class[c].to_dict() for c in class_names}})
top = per_class.loc[global_imp.index[:20]].assign(overall=global_imp[:20])
put_text(shap_dir + "report.html", "<html><head><meta charset='utf-8'><title>SHAP report</title></head><body>"
         f"<h1>SHAP explainability (mean |SHAP|, {len(sample)} test windows)</h1>"
         f"<p>Exact TreeSHAP on {PKG_ARN}</p>" + top.round(4).to_html() + "</body></html>")
print("SHAP output:", shap_dir)
display(top.round(4))
global_imp[:20][::-1].plot.barh(figsize=(7, 6), title="Top 20 features - mean |SHAP| over classes");

### Consolidated report index

In [ ]:
dq, mq = latest_subprefix(MON("data-quality/runs/")), latest_subprefix(MON("model-quality/runs/"))
shap_run = latest_subprefix(SHAP_OUT)
debugger = train_info.get("debugger_artifacts")
cands = {
    "data_quality_baseline_statistics": MON("data-quality/baseline/statistics.json"),
    "data_quality_baseline_constraints": MON("data-quality/baseline/constraints.json"),
    "data_quality_violations": dq and dq + "constraint_violations.json",
    "data_quality_violations_summary": dq and dq + "violations_summary.json",
    "model_quality_baseline_statistics": MON("model-quality/baseline/statistics.json"),
    "model_quality_baseline_constraints": MON("model-quality/baseline/constraints.json"),
    "model_quality_violations": mq and mq + "constraint_violations.json",
    "model_quality_violations_summary": mq and mq + "violations_summary.json",
    "shap_analysis": shap_run and shap_run + "analysis.json",
    "shap_report_html": shap_run and shap_run + "report.html",
    "debugger_report": debugger and debugger + M.DEBUGGER_REPORT_PATH,
}
# a violations file legitimately absent means zero violations; keep the summary as the evidence
cands = {k: (v if v and exists(v) else None) for k, v in cands.items()}
index = M.report_index(cands)
write_json(MON("reports/index.json"), index)
display(pd.DataFrame([{"report": k, "uri": v, "exists": exists(v)} for k, v in index["reports"].items()]))
print("Not found (job did not emit them or has not run):", index["missing"])


## Cleanup

Runs once for all sections above: nothing here creates schedules or endpoints; lists them, deletes only temporary pamap2-har-mon* models, warns about still-running jobs.

In [ ]:
# Cleanup: nothing here creates schedules or endpoints; list them, delete only temporary pamap2-har-mon* models.
for page in sm.get_paginator("list_monitoring_schedules").paginate(NameContains=cfg["prefix"]):
    for x in page["MonitoringScheduleSummaries"]:
        print("WARNING: monitoring schedule exists (not deleted):", x["MonitoringScheduleName"])
for page in sm.get_paginator("list_models").paginate(NameContains=f"{cfg['prefix']}-mon"):
    for m in page["Models"]:
        if m["ModelName"].startswith(f"{cfg['prefix']}-mon"):
            print("Deleting temporary model", m["ModelName"])
            sm.delete_model(ModelName=m["ModelName"])
for kind, pager, key, field in (("processing", "list_processing_jobs", "ProcessingJobSummaries", "ProcessingJobName"),
                                ("transform", "list_transform_jobs", "TransformJobSummaries", "TransformJobName")):
    for page in sm.get_paginator(pager).paginate(StatusEquals="InProgress", NameContains=cfg["prefix"]):
        for x in page[key]:
            print("WARNING: still running", kind, "job:", x[field])
print("Endpoints:", [e["EndpointName"] for p in sm.get_paginator("list_endpoints").paginate() for e in p["Endpoints"]])